In [1]:
!pip install scikit-image


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.1.2 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import cv2
import openslide
import numpy as np
from PIL import Image
from tqdm import tqdm
from skimage.measure import shannon_entropy


In [3]:
PATCH_SIZE  = 224
STRIDE      = 224
LEVEL       = 0
MASK_LEVEL  = 2
MAX_PATCHES = 700


In [4]:
def load_slide(slide_path):
    assert os.path.exists(slide_path), "Slide path does not exist"
    return openslide.OpenSlide(slide_path)


In [5]:
def create_tissue_mask(slide, thresh=200):
    img = np.array(
        slide.read_region((0, 0), MASK_LEVEL, slide.level_dimensions[MASK_LEVEL])
    )[:, :, :3]

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)
    return mask


In [6]:
def score_patch(slide, coord, tissue_mask):
    x, y = coord

    scale = slide.level_downsamples[MASK_LEVEL]
    mx, my = int(x / scale), int(y / scale)
    msize  = int(PATCH_SIZE / scale)

    mask_patch = tissue_mask[my:my + msize, mx:mx + msize]
    tissue_ratio = np.mean(mask_patch > 0)

    if tissue_ratio < 0.6:
        return None

    patch = slide.read_region((x, y), LEVEL, (PATCH_SIZE, PATCH_SIZE)).convert("RGB")
    patch_np = np.array(patch)
    gray = cv2.cvtColor(patch_np, cv2.COLOR_RGB2GRAY)

    entropy = shannon_entropy(gray)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.mean(edges > 0)

    score = (0.5 * entropy) + (0.3 * edge_density) + (0.2 * tissue_ratio)
    return score, patch


In [7]:
def extract_best_patches(slide):
    tissue_mask = create_tissue_mask(slide)
    w, h = slide.level_dimensions[LEVEL]

    scored = []

    for y in tqdm(range(0, h - PATCH_SIZE, STRIDE)):
        for x in range(0, w - PATCH_SIZE, STRIDE):
            result = score_patch(slide, (x, y), tissue_mask)
            if result is not None:
                scored.append(result)

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:MAX_PATCHES]


In [8]:
def save_patches(patches, output_path):
    os.makedirs(output_path, exist_ok=True)

    for i, (_, patch) in enumerate(patches):
        patch.save(os.path.join(output_path, f"patch_{i}.png"))


In [9]:
def extract_patches_from_slide(slide_path, output_path):
    print("Loading slide...")
    slide = load_slide(slide_path)

    print("Extracting best patches...")
    patches = extract_best_patches(slide)

    print(f"Saving {len(patches)} patches to:", output_path)
    save_patches(patches, output_path)

    print("Done.")


In [17]:
slide_path  = "hcc_down/TCGA-W5-AA2T-01Z-00-DX1.0DC8411E-9AAA-4DA0-AF3D-E1E2A59AF229.svs"
output_path = "TEST/chol/chol-1"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 377/377 [12:24<00:00,  1.97s/it]


Saving 700 patches to: TEST/chol/chol-1
Done.


In [18]:
slide_path  = "hcc_down/TCGA-W5-AA30-01Z-00-DX1.1B63C128-3C86-4ED2-9A26-15BB5E47D66A.svs"
output_path = "TEST/chol/chol-2"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 383/383 [14:36<00:00,  2.29s/it]


Saving 700 patches to: TEST/chol/chol-2
Done.


In [19]:
slide_path  = "hcc_down/TCGA-ZH-A8Y1-01Z-00-DX1.2FE51BA7-9B4D-48B3-AB81-9C744DBA13A4.svs"
output_path = "TEST/chol/chol-3"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 322/322 [15:47<00:00,  2.94s/it]


Saving 700 patches to: TEST/chol/chol-3
Done.


In [20]:
slide_path  = "hcc_down/TCGA-ZH-A8Y3-01Z-00-DX1.D4110867-EC54-4455-A335-3CB51AF67908.svs"
output_path = "TEST/chol/chol-4"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 337/337 [16:15<00:00,  2.90s/it]


Saving 700 patches to: TEST/chol/chol-4
Done.


In [21]:
slide_path  = "hcc_down/TCGA-ZH-A8Y4-01Z-00-DX1.80DEF07F-1DDD-4998-82FC-C5CC68904D32.svs"
output_path = "TEST/chol/chol-5"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 382/382 [11:54<00:00,  1.87s/it]


Saving 700 patches to: TEST/chol/chol-5
Done.


In [24]:
slide_path  = "hcc_down/TCGA-ZH-A8Y5-01Z-00-DX1.A7172CB3-474B-455E-8F62-DDE31CAA6783.svs"
output_path = "TEST/chol/chol-6"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 378/378 [12:41<00:00,  2.02s/it]


Saving 700 patches to: TEST/chol/chol-6
Done.


In [22]:
slide_path  = "hcc_down/TCGA-W5-AA2W-01Z-00-DX1.8EF8CC01-E6E2-4197-BDE7-6184D327A161.svs"
output_path = "TEST/chol/chol-7"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 409/409 [12:13<00:00,  1.79s/it]


Saving 700 patches to: TEST/chol/chol-7
Done.


In [25]:
slide_path  = "hcc_down/TCGA-ZH-A8Y8-01Z-00-DX1.174309BD-9629-4567-9646-A18CD9D54FDE.svs"
output_path = "TEST/chol/chol-8"
os.makedirs(output_path, exist_ok=True)

extract_patches_from_slide(slide_path, output_path)

Loading slide...
Extracting best patches...


100%|██████████| 350/350 [12:47<00:00,  2.19s/it]


Saving 700 patches to: TEST/chol/chol-8
Done.
